# Trustworthy Medical AI Under Real-World Uncertainty

Healthcare AI module prototype for the **AI & Uncertainty School and Workshop 2026**.

This Colab notebook is designed to test the teaching flow before the final workshop materials are prepared. It uses synthetic EHR-like data so everything runs immediately without protected-data access.

## Teaching flow
1. Bayesian updating and posterior uncertainty
2. AUROC versus calibration
3. Epistemic versus aleatoric uncertainty
4. Distribution shift across hospitals
5. Informative missingness
6. Ensemble uncertainty and abstention
7. Hierarchical thinking
8. Simulation-based inference bridge
9. Team project with hidden failure modes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.stats import beta
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, accuracy_score
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True


## 1. Bayesian updating: prior to posterior
A point estimate hides uncertainty. The posterior distribution shows both the estimate and how uncertain we remain.


In [ ]:
prior_a, prior_b = 2, 2
successes, failures = 8, 2
post_a, post_b = prior_a + successes, prior_b + failures
p = np.linspace(0, 1, 500)
plt.plot(p, beta.pdf(p, prior_a, prior_b), label='Prior')
plt.plot(p, beta.pdf(p, post_a, post_b), label='Posterior')
plt.axvline(post_a/(post_a+post_b), linestyle='--', label='Posterior mean')
plt.xlabel('Probability the clinical alert is useful')
plt.ylabel('Density')
plt.title('Bayesian updating: evidence narrows uncertainty')
plt.legend(); plt.show()
print('Posterior mean:', round(post_a/(post_a+post_b), 3))
print('95% credible interval:', np.round(beta.ppf([0.025, 0.975], post_a, post_b), 3))


## 2. Same ranking, different calibration
AUROC measures ranking. Calibration asks whether predicted probabilities mean what they claim to mean.


In [ ]:
n = 4000
latent_risk = rng.normal(0, 1, n)
true_p = expit(latent_risk)
y = rng.binomial(1, true_p)
p_good = np.clip(true_p + rng.normal(0, 0.03, n), 0.001, 0.999)
logit = np.log(p_good/(1-p_good))
p_over = expit(2.2*logit)
for name, pred in [('Reasonably calibrated', p_good), ('Overconfident', p_over)]:
    print(name, '| AUROC =', round(roc_auc_score(y,pred),3), '| Brier =', round(brier_score_loss(y,pred),3), '| Log loss =', round(log_loss(y,pred),3))
    frac_pos, mean_pred = calibration_curve(y, pred, n_bins=10, strategy='quantile')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)
plt.plot([0,1],[0,1], linestyle='--', label='Perfect calibration')
plt.xlabel('Predicted risk'); plt.ylabel('Observed event rate')
plt.title('AUROC can look good while probabilities are wrong')
plt.legend(); plt.show()


## 3. Epistemic versus aleatoric uncertainty


In [ ]:
x = np.linspace(-4, 4, 250)
mean = expit(1.2*x)
aleatoric = mean*(1-mean)
epistemic = 0.02 + 0.20*(np.abs(x)/np.max(np.abs(x)))**2
plt.plot(x, mean, label='Predicted event probability')
plt.fill_between(x, np.clip(mean-epistemic,0,1), np.clip(mean+epistemic,0,1), alpha=0.25, label='Illustrative epistemic band')
plt.xlabel('Simplified patient risk score'); plt.ylabel('Probability')
plt.title('Epistemic uncertainty grows outside familiar regions')
plt.legend(); plt.show()
plt.plot(x, aleatoric, label='Aleatoric uncertainty')
plt.plot(x, epistemic, label='Epistemic uncertainty')
plt.xlabel('Simplified patient risk score'); plt.ylabel('Illustrative uncertainty')
plt.title('Two different uncertainty sources')
plt.legend(); plt.show()


## 4. Synthetic EHR-like simulator
The simulator creates realistic failure modes without requiring protected clinical data.


In [ ]:
def simulate_ehr(n=5000, site='A', prevalence_shift=0.0, measurement_noise=1.0, missingness_strength=0.0, label_noise=0.0, subgroup_shift=0.0, seed=42):
    r = np.random.default_rng(seed)
    age = np.clip(r.normal(66,14,n),18,95)
    heart_rate = r.normal(82,15,n)
    systolic_bp = r.normal(128,22,n)
    creatinine = np.exp(r.normal(np.log(1.0),0.45,n))
    prior_stroke = r.binomial(1,0.18,n)
    diabetes = r.binomial(1,0.28,n)
    group = r.binomial(1,0.35,n)
    site_effect = {'A':0.0,'B':0.35,'C':-0.30}.get(site,0.0)
    linear = -5.1 + 0.035*age + 0.018*(heart_rate-80) - 0.012*(systolic_bp-125) + 0.65*np.log(creatinine) + 0.70*prior_stroke + 0.50*diabetes + subgroup_shift*group + site_effect + prevalence_shift
    p_event = expit(linear)
    y = r.binomial(1,p_event)
    flip = r.random(n) < label_noise
    y = np.where(flip,1-y,y)
    df = pd.DataFrame({'age':age,'heart_rate':heart_rate+r.normal(0,3*measurement_noise,n),'systolic_bp':systolic_bp+r.normal(0,5*measurement_noise,n),'creatinine':np.maximum(0.2,creatinine+r.normal(0,0.08*measurement_noise,n)),'prior_stroke':prior_stroke,'diabetes':diabetes,'group':group,'site':site,'outcome':y,'true_risk':p_event})
    if missingness_strength > 0:
        miss_p = expit(-3.0 + missingness_strength*(0.7*(heart_rate>95) + 0.8*(creatinine>1.4)))
        df.loc[r.random(n)<miss_p,'heart_rate'] = np.nan
        df.loc[r.random(n)<miss_p,'creatinine'] = np.nan
    return df
simulate_ehr().head()


## 5. Distribution shift across hospitals


In [ ]:
features = ['age','heart_rate','systolic_bp','creatinine','prior_stroke','diabetes','group']
train = simulate_ehr(n=5000, site='A', seed=1)
site_b = simulate_ehr(n=3000, site='B', prevalence_shift=0.5, seed=2)
site_c = simulate_ehr(n=3000, site='C', measurement_noise=1.8, seed=3)
model = make_pipeline(SimpleImputer(strategy='median'), LogisticRegression(max_iter=1000))
model.fit(train[features], train['outcome'])
def evaluate(name,data):
    p = model.predict_proba(data[features])[:,1]
    return {'dataset':name,'event_rate':data['outcome'].mean(),'AUROC':roc_auc_score(data['outcome'],p),'Brier':brier_score_loss(data['outcome'],p),'LogLoss':log_loss(data['outcome'],p)}
display(pd.DataFrame([evaluate('Site A',train),evaluate('Site B',site_b),evaluate('Site C',site_c)]).round(3))
for name,data in [('Site A',train),('Site B',site_b),('Site C',site_c)]:
    p = model.predict_proba(data[features])[:,1]
    frac_pos, mean_pred = calibration_curve(data['outcome'], p, n_bins=8, strategy='quantile')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)
plt.plot([0,1],[0,1], linestyle='--')
plt.xlabel('Predicted risk'); plt.ylabel('Observed event rate')
plt.title('Model probabilities can change meaning across hospitals')
plt.legend(); plt.show()


## 6. Informative missingness


In [ ]:
missing_df = simulate_ehr(n=5000, missingness_strength=1.4, seed=10)
missing_df['creatinine_missing'] = missing_df['creatinine'].isna().astype(int)
summary = missing_df.groupby('creatinine_missing')['outcome'].agg(['mean','count'])
summary.index = ['Creatinine observed','Creatinine missing']
display(summary)
summary['mean'].plot(kind='bar')
plt.ylabel('Observed event rate'); plt.title('Missingness can carry clinical information')
plt.xticks(rotation=0); plt.show()


# Lecture 2: From prediction to trustworthy clinical use
## 7. Ensemble uncertainty and selective prediction


In [ ]:
train_e = simulate_ehr(n=2500, site='A', seed=22)
X = train_e[features]; y_e = train_e['outcome']
imp = SimpleImputer(strategy='median'); X_imp = imp.fit_transform(X)
ensemble_probs = []
for b in range(40):
    idx = rng.integers(0,len(train_e),len(train_e))
    m = LogisticRegression(max_iter=1000)
    m.fit(X_imp[idx], y_e.iloc[idx])
    ensemble_probs.append(m.predict_proba(X_imp)[:,1])
ensemble_probs = np.vstack(ensemble_probs)
mean_p = ensemble_probs.mean(axis=0); sd_p = ensemble_probs.std(axis=0)
plt.scatter(mean_p, sd_p, alpha=0.35)
plt.xlabel('Mean predicted risk'); plt.ylabel('Across-model SD')
plt.title('Bootstrap ensemble as a simple epistemic uncertainty proxy')
plt.show()
order = np.argsort(sd_p)
coverages = np.linspace(0.2,1.0,17); risk = []
for coverage in coverages:
    k = int(len(order)*coverage); keep = order[:k]
    pred = (mean_p[keep]>=0.5).astype(int)
    risk.append(1-accuracy_score(y_e.iloc[keep],pred))
plt.plot(coverages,risk,marker='o')
plt.xlabel('Coverage: fraction receiving an automated prediction')
plt.ylabel('Error rate among retained predictions')
plt.title('Abstention trades coverage for reliability')
plt.show()


## 8. Hierarchical thinking: partial pooling across sites


In [ ]:
site_summary = pd.DataFrame({'site':['Large hospital','Medium hospital','Small hospital','Tiny hospital'],'events':[210,42,8,1],'patients':[1000,180,30,4]})
a0,b0 = 2,8
site_summary['raw_rate'] = site_summary['events']/site_summary['patients']
site_summary['posterior_mean'] = (site_summary['events']+a0)/(site_summary['patients']+a0+b0)
display(site_summary)
x = np.arange(len(site_summary))
plt.scatter(x, site_summary['raw_rate'], s=90, label='Raw site estimate')
plt.scatter(x, site_summary['posterior_mean'], s=90, label='Partially pooled estimate')
plt.xticks(x, site_summary['site'], rotation=20)
plt.ylabel('Estimated event probability')
plt.title('Small sites shrink more toward the shared prior')
plt.legend(); plt.show()


## 9. Simulation-based inference bridge
In healthcare, we can define a simulator that generates observed data from latent parameters, then infer which parameter values plausibly generated the observed data.


In [ ]:
def simulator(theta,n=250,seed=None):
    r = np.random.default_rng(seed)
    age = np.clip(r.normal(65,14,n),18,95)
    p = expit(-4.3 + 0.04*age + theta)
    y = r.binomial(1,p)
    return {'event_rate':y.mean()}
theta_true = 0.55
observed = simulator(theta_true,n=600,seed=123)
n_sims = 8000
theta_prior = rng.uniform(-1.5,1.5,n_sims)
distances = []
for i,theta in enumerate(theta_prior):
    s = simulator(theta,n=600,seed=1000+i)
    distances.append(abs(s['event_rate']-observed['event_rate']))
distances = np.array(distances)
accepted = theta_prior[distances <= np.quantile(distances,0.03)]
plt.hist(theta_prior,bins=50,density=True,alpha=0.35,label='Prior')
plt.hist(accepted,bins=35,density=True,alpha=0.60,label='ABC posterior')
plt.axvline(theta_true,linestyle='--',label='True latent value')
plt.xlabel('Latent risk parameter θ'); plt.ylabel('Density')
plt.title('Simulation-based inference: invert a forward simulator')
plt.legend(); plt.show()


# Team project prototype: same task, different hidden failure modes

**Core question:** Can your team build a clinical prediction system that knows when it should not be trusted?

Each team receives what appears to be the same prediction task, but its dataset contains a different hidden problem:
1. informative missingness
2. site/prevalence shift
3. measurement noise
4. label noise
5. subgroup shift
6. out-of-distribution patients

The objective is not to maximize AUROC. The team must diagnose the problem, quantify uncertainty, inspect calibration, and define when the model should abstain or defer.


In [ ]:
team_datasets = {
    'Team 1 - Missingness': simulate_ehr(n=3000, missingness_strength=1.5, seed=101),
    'Team 2 - Site shift': simulate_ehr(n=3000, site='B', prevalence_shift=0.5, seed=102),
    'Team 3 - Measurement noise': simulate_ehr(n=3000, measurement_noise=2.5, seed=103),
    'Team 4 - Label noise': simulate_ehr(n=3000, label_noise=0.12, seed=104),
    'Team 5 - Subgroup shift': simulate_ehr(n=3000, subgroup_shift=1.0, seed=105)
}
for name,d in team_datasets.items():
    print(name, '| event rate:', round(d['outcome'].mean(),3), '| missing cells:', int(d[features].isna().sum().sum()))


## Team deliverable
Each team gives a **5-minute presentation** answering:
1. What uncertainty or failure mode did you discover?
2. Which metric or visualization exposed it?
3. Did the model remain calibrated?
4. Which patients or situations were least trustworthy?
5. When should the model abstain or defer?
6. What additional data would reduce uncertainty?
7. Would you deploy this model? Under what conditions?
